## **LangChain** - _Parsers_

Hasta ahora, hemos recibido respuestas de **Gemini** en formato de texto libre (_strings_). Esto está muy bien si estamos construyendo un chatbot para que lo lea un humano, pero **¿qué pasa si queremos que nuestro código lea esa respuesta?**

Si le pedimos al **LLM** que analice un correo electrónico y extraiga el nombre del remitente y el nivel de urgencia para guardarlo en una base de datos, no nos sirve que responda: *"¡Claro! El nombre del remitente es Juan y la urgencia es alta. ¿Te ayudo con algo más?"*. Necesitamos un **JSON** estricto o un objeto de **Python**.

Aquí es donde entran los **Output Parsers**.

In [1]:
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# Variables de entorno
load_dotenv()

MODEL = "gemini-3.1-flash-lite"
TEMPERATURE = 0

# Instanciamos el modelo asegurando temperature=0 para respuestas lógicas y deterministas
llm = ChatGoogleGenerativeAI(model = MODEL, temperature = TEMPERATURE)

c:\Users\danie\OneDrive\Documentos\GitHub\DAIXXRT\LangChain - Boost Academy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 01. **StrOutputParser**

Cuando invocamos un **LLM** en **LangChain**, la respuesta cruda no es un simple texto, sino un objeto estructurado llamado `AIMessage`. Este objeto contiene el texto de la respuesta, pero también metadatos de la **API**, conteo de tokens, entre otros.

El `StrOutputParser` es el parser más sencillo y la pieza fundamental de casi cualquier cadena básica. Su único trabajo es tomar ese objeto `AIMessage` y extraer limpiamente el texto de la respuesta (su atributo `.content`), entregándonos un **string** de **Python** listo para usar.

In [2]:
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(template = "Dime una frase motivacional muy corta sobre {tema}.")

# Cadena SIN parser
cadena_cruda = prompt | llm
resultado_crudo = cadena_cruda.invoke(input = {"tema": "aprender Python"})

print(f"Tipo de dato: {type(resultado_crudo)}")
print(f"Contenido crudo: {resultado_crudo}")

Tipo de dato: <class 'langchain_core.messages.ai.AIMessage'>
Contenido crudo: content=[{'type': 'text', 'text': '**"Escribe código hoy, construye tu futuro mañana."**', 'extras': {'signature': 'EjQKMgEMOdbHrjXdyifQRnCnivLeyxgP+SbF8108ifLTEo1FcgiJViqNTIMJ6gsfMiM3lpXD'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019eb10a-560b-7c93-a4fd-5dc0af13cc94-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 13, 'output_tokens': 13, 'total_tokens': 26, 'input_token_details': {'cache_read': 0}}


In [3]:
# Añadimos el StrOutputParser
parser = StrOutputParser()
cadena_limpia = prompt | llm | parser

resultado_limpio = cadena_limpia.invoke(input = {"tema": "aprender Python"})

print(f"Tipo de dato: {type(resultado_limpio)}")
print(f"Contenido limpio: {resultado_limpio}")

Tipo de dato: <class 'langchain_core.messages.base.TextAccessor'>
Contenido limpio: **"Escribe código hoy, construye tu futuro mañana."**


### 02. **CommaSeparatedListOutputParser**

Muchas veces necesitamos que el modelo nos devuelva un listado de elementos simples (por ejemplo: "nombra 5 lenguajes de programación").

Para estos casos utilizamos el `CommaSeparatedListOutputParser`:
- Nos genera unas instrucciones de formato (`get_format_instructions()`) para pedirle al LLM que separe los elementos por comas.
- Toma la respuesta de texto, la divide por las comas, elimina los espacios en blanco sobrantes y nos devuelve una lista de **Python** (`list`).

In [4]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# Instanciamos el parser
parser = CommaSeparatedListOutputParser()

# Creamos el prompt, necesitamos inyectar las instrucciones de formato en el prompr
prompt = ChatPromptTemplate.from_template(template = "Nombra {cantidad} {tema}.\\n\\n{instrucciones_formato}")

# Construimos la cadena
cadena_lista = prompt | llm | parser

# Ejecutamos pasándole las variables y las instrucciones generadas por el parser
llm_input = {"cantidad" : 5,
             "tema" : "conceptos clave de bases de datos relacionales",
             "instrucciones_formato" : parser.get_format_instructions()}

resultado_lista = cadena_lista.invoke(input = llm_input)

print("Resultado obtenido:", resultado_lista)
print("Tipo de dato:", type(resultado_lista))

Resultado obtenido: ['Tabla', 'Clave primaria', 'Clave foránea', 'Relación', 'Normalización']
Tipo de dato: <class 'list'>


In [7]:
resultado_lista

['Tabla', 'Clave primaria', 'Clave foránea', 'Relación', 'Normalización']

In [8]:
parser.get_format_instructions()

'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'

### 03. **JSON**

Podemos intentar pedirle **JSON** al modelo directamente en el prompt, pero los **LLMs** tienen suelen envolver el **JSON** en bloques de código markdown (\```json ... ```) o añadir texto conversacional antes o después.

Veamos un ejemplo de este caso:

In [9]:
prompt = ChatPromptTemplate.from_template(template = "Extrae el nombre y la ciudad de este texto: '{texto}'. Devuélvelo en formato JSON.")

cadena_ingenua = prompt | llm

llm_input = {"texto": "Me llamo Daniel y te escribo desde Madrid."}

respuesta = cadena_ingenua.invoke(input = llm_input)

print(respuesta.content[0]["text"])
print("Comparamos tipos de datos:", type(respuesta.content) == dict)

```json
{
  "nombre": "Daniel",
  "ciudad": "Madrid"
}
```
Comparamos tipos de datos: False


In [22]:
eval(respuesta.content[0]["text"].strip("`").replace("\n", "").strip("json"))

{'nombre': 'Daniel', 'ciudad': 'Madrid'}

### 04. **JsonOutputParser**

**LangChain** soluciona esto con `JsonOutputParser`. Este componente se encarga de:
1. **Te da instrucciones de formato (`get_format_instructions()`):** Genera un texto súper específico para inyectar en tu prompt, explicándole al **LLM** exactamente cómo debe formatear su salida para no fallar.
2. **Limpia y transforma:** Toma la respuesta cruda del modelo, elimina el markdown sobrante y usa `json.loads()` de **Python** para retornar un diccionario (`dict`).

In [23]:
from langchain_core.output_parsers import JsonOutputParser

# Instanciamos el parser
parser_json = JsonOutputParser()

# Creamos el prompt con las instrucciones del parser
prompt_json = ChatPromptTemplate.from_template(template = "Extrae el nombre y la ciudad de este texto: '{texto}'.\n\n{instrucciones_formato}")

# Construimos la cadena
cadena_json = prompt_json | llm | parser_json

# Ejecutamos pasándole las variables. 
# Debemos pasar las instrucciones de formato generadas por el parser.

llm_input = {"texto" : "Me llamo Daniel y te escribo desde Madrid.",
             "instrucciones_formato" : parser_json.get_format_instructions()}

resultado_json = cadena_json.invoke(input = llm_input)

print(resultado_json)
print("Comparamos tipos de datos:", isinstance(resultado_json, dict))
print("Nombre extraído:", resultado_json["nombre"])
print("Ciudad extraída:", resultado_json["ciudad"])

{'nombre': 'Daniel', 'ciudad': 'Madrid'}
Comparamos tipos de datos: True
Nombre extraído: Daniel
Ciudad extraída: Madrid


In [24]:
parser_json.get_format_instructions()

'Return a JSON object.'

### 05. **PydanticOutputParser**

El `JsonOutputParser` es increible, pero tiene un problema: ¿qué pasa si el modelo decide llamar a la llave "Ciudad" en lugar de "ciudad" (con minúscula)?.

Para aplicaciones en producción, usamos **Pydantic**, la librería estándar de validación de datos en Python. Nos permite definir una "Clase" con los tipos de datos exactos que esperamos (ej. el nombre debe ser un `string`, la edad un `integer`). Si el **LLM** se equivoca, el parser lanza un error o intenta corregirlo.

In [25]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# Definimos nuestro esquema estricto de datos con Pydantic
class PerfilUsuario(BaseModel):
    nombre: str = Field(description = "El nombre de la persona")
    edad: int = Field(description = "La edad de la persona en números enteros")
    habilidades: list[str] = Field(description = "Lista de habilidades profesionales mencionadas")

# Instanciamos el parser atado a nuestra clase
parser_pydantic = PydanticOutputParser(pydantic_object = PerfilUsuario)

# Creamos el prompt (nota cómo inyectamos las instrucciones igual que antes)
prompt_pydantic = ChatPromptTemplate.from_template(template = """Analiza el siguiente currículum y extrae la información requerida.\n\n
                                                                 Currículum: {curriculum}\n\n
                                                                 {instrucciones_formato}""")

# Construimos la cadena
cadena_pydantic = prompt_pydantic | llm | parser_pydantic

# Ejecutamos
texto_cv = "Soy Ana, tengo 28 años y soy experta en Python, SQL y diseño de bases de datos. Busco nuevos retos."

llm_input = {"curriculum": texto_cv,
             "instrucciones_formato": parser_pydantic.get_format_instructions()}

perfil_extraido = cadena_pydantic.invoke(input = llm_input)

print(perfil_extraido)
print("Tipo de dato:", type(perfil_extraido))
print(f"La candidata {perfil_extraido.nombre} tiene {len(perfil_extraido.habilidades)} habilidades principales.")

nombre='Ana' edad=28 habilidades=['Python', 'SQL', 'diseño de bases de datos']
Tipo de dato: <class '__main__.PerfilUsuario'>
La candidata Ana tiene 3 habilidades principales.


In [26]:
parser_pydantic.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"nombre": {"description": "El nombre de la persona", "title": "Nombre", "type": "string"}, "edad": {"description": "La edad de la persona en números enteros", "title": "Edad", "type": "integer"}, "habilidades": {"description": "Lista de habilidades profesionales mencionadas", "items": {"type": "string"}, "title": "Habilidades", "type": "array"}}, "required": ["nombre", "edad", "habilidades"]}\n```'

In [27]:
perfil_extraido

PerfilUsuario(nombre='Ana', edad=28, habilidades=['Python', 'SQL', 'diseño de bases de datos'])

In [31]:
# Podemos usar el método .model_dump() para obtener un diccionario del objeto de tipo PerfilUsuario

perfil_extraido.model_dump()

{'nombre': 'Ana',
 'edad': 28,
 'habilidades': ['Python', 'SQL', 'diseño de bases de datos']}

In [32]:
import pandas as pd
pd.json_normalize(perfil_extraido.model_dump())

,nombre,edad,habilidades
0,Ana,28,"[Python, SQL, diseño de bases de datos]"


### 06. **LLM Native Structured Output**

En las secciones anteriores vimos cómo usar `JsonOutputParser` y `PydanticOutputParser`. Ambos métodos funcionan bajo el mismo principio histórico: **Prompt Injection**. Es decir, **LangChain** oculta un texto largo en `{instrucciones_formato}` rogándole al modelo que responda en **JSON** y que no agregue saludos ni texto extra.

Sin embargo, los modelos modernos (como **Gemini**, **OpenAI** o **Anthropic**) ya están entrenados para entender esquemas de datos directamente desde su **API**. No necesitamos darles las instrucciones a través del prompt; la propia **API** del proveedor fuerza al modelo a devolver la estructura exacta.

**LangChain** unifica esto con el método `.with_structured_output()`.

**Ventajas:**
1. **Prompts más limpios:** Ya no necesitas inyectar variables de instrucciones de formato.
2. **Mayor eficiencia:** Ahorras tokens de entrada (al eliminar las instrucciones largas).
3. **Robustez nativa:** El modelo reduce drásticamente las alucinaciones de formato porque el motor de inferencia está limitado a tu esquema.

In [33]:
# Repetimos el mismo ejemplo
# Modificamos el modelo y el prompt

# Esto configura la API de Gemini para que solo sepa responder con este formato
llm_estructurado = llm.with_structured_output(schema = PerfilUsuario)

# Creamos el prompt (nota cómo NO inyectamos las instrucciones)
prompt = ChatPromptTemplate.from_template(template = """Analiza el siguiente currículum y extrae la información requerida.\n\n
                                                                 Currículum: {curriculum}""")

# Construimos la cadena, omitimos el parser
cadena_pydantic = prompt | llm_estructurado

# Ejecutamos
texto_cv = "Soy Ana, tengo 28 años y soy experta en Python, SQL y diseño de bases de datos. Busco nuevos retos."

llm_input = {"curriculum": texto_cv}

perfil_extraido = cadena_pydantic.invoke(input = llm_input)

# La salida del modelo es directamente de tipo PerfilUsuario 

print(perfil_extraido)
print("Tipo de dato:", type(perfil_extraido))
print(f"La candidata {perfil_extraido.nombre} tiene {len(perfil_extraido.habilidades)} habilidades principales.")

nombre='Ana' edad=28 habilidades=['Python', 'SQL', 'diseño de bases de datos']
Tipo de dato: <class '__main__.PerfilUsuario'>
La candidata Ana tiene 3 habilidades principales.


In [34]:
perfil_extraido

PerfilUsuario(nombre='Ana', edad=28, habilidades=['Python', 'SQL', 'diseño de bases de datos'])

In [35]:
# Podemos usar el método .model_dump() para obtener un diccionario del objeto de tipo PerfilUsuario

perfil_extraido.model_dump()

{'nombre': 'Ana',
 'edad': 28,
 'habilidades': ['Python', 'SQL', 'diseño de bases de datos']}